## **ONNX Export & Inference Benchmarking**

**CONCEPT: Why export to ONNX?**

Your trained model is a PyTorch .pt file. PyTorch is a training framework — it has a lot of overhead for automatic differentiation, gradient tracking, etc. that you don't need at inference time. ONNX (Open Neural Network Exchange) is a standardized model format that can be run by ONNX Runtime, a highly optimized inference engine. ONNX Runtime strips out all training overhead, uses hardware-specific optimizations (SIMD, quantization, graph fusion), and runs 2-4x faster than PyTorch inference.For a real-time safety system, this difference between 18 FPS and 35 FPS is the difference between
real-time monitoring and a slideshow. This is the engineering decision that impresses interviewers.

### Export to ONNX

In [1]:
from ultralytics import YOLO

model = YOLO(r'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.pt')

# Export options:
# format='onnx' — target format
# imgsz=640 — input size must match training size
# dynamic=False — fixed batch size 1 (simpler, faster for inference API)
# simplify=True — runs onnx-simplifier to remove redundant operations

export_path = model.export(format = 'onnx', imgsz = 640, dynamic = False, simplify = True)
print(f'Exported to: {export_path}')

Ultralytics 8.4.56  Python-3.10.20 torch-2.7.1+cu118 CPU (13th Gen Intel Core i7-13700F)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs

PyTorch: starting from 'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 14, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71'] not found, attempting AutoUpdate...
Using Python 3.10.20 environment at: c:\Users\user\.conda\envs\safety-cv
Resolved 10 packages in 779ms
 Downloaded onnx
Prepared 3 packages in 6.40s
Installed 3 packages in 1.56s
 + ml-dtypes==0.5.4
 + onnx==1.21.0
 + onnxslim==0.1.94

requirements: AutoUpdate success  9.9s
WARNING requirements: Restart runtime or rerun command for updates to

c:\Users\user\.conda\envs\safety-cv\lib\site-packages\ultralytics\nn\modules\head.py:189: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.dynamic or self.shape != shape:


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  11.4s, saved as 'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.onnx' (42.7 MB)

Export complete (11.8s)
Results saved to C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.onnx imgsz=640 data=C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\dataset\Construction Site Safety.v27-yolov8.yolov8\data.yaml  
Visualize:       https://netron.app
Exported to: C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\w

In [4]:
# Cell 2 — benchmark
import time, numpy as np
import onnxruntime as ort
import torch
from ultralytics import YOLO

NUM_RUNS = 100
DEVICE = 'cpu' # Forces stable execution path

# ── 1. FIX DUMMY INPUT SHAPE ──────────────────────────────────────────
# PyTorch/YOLO wants individual images shaped as (H, W, C) -> (640, 640, 3)
# Values scaled to 0-255 to mimic standard float photographic inputs
dummy_input_img = (np.random.rand(640, 640, 3) * 255.0).astype(np.float32)

# ONNX Runtime expectations are different; it requires a 4D batch array (B, C, H, W)
# We transpose our (640, 640, 3) image to (3, 640, 640) and append a batch dimension
dummy_input_onnx = np.expand_dims(dummy_input_img.transpose(2, 0, 1), axis=0)

# ── PyTorch Benchmark ──────────────────────────────────────────────────
pt_model = YOLO(r'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.pt')

# Warmup 
for _ in range(5):
    # Pass the standard (640, 640, 3) format to the PyTorch pipeline
    _ = pt_model(dummy_input_img, verbose=False, device=DEVICE)

start = time.perf_counter()
for _ in range(NUM_RUNS):
    _ = pt_model(dummy_input_img, verbose=False, device=DEVICE)
pt_time = (time.perf_counter() - start) / NUM_RUNS * 1000
pt_fps = 1000 / pt_time

# ── ONNX Runtime Benchmark ─────────────────────────────────────────────
if DEVICE == 'cpu':
    providers = ['CPUExecutionProvider']
else:
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

sess = ort.InferenceSession(
    r'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.onnx',
    providers=providers
)
input_name = sess.get_inputs()[0].name

# Warmup
for _ in range(5):
    # Pass the matching 4D tensor (1, 3, 640, 640) directly to the ONNX session
    _ = sess.run(None, {input_name: dummy_input_onnx})

start = time.perf_counter()
for _ in range(NUM_RUNS):
    _ = sess.run(None, {input_name: dummy_input_onnx})
onnx_time = (time.perf_counter() - start) / NUM_RUNS * 1000
onnx_fps = 1000 / onnx_time

# ── Print Results ──────────────────────────────────────────────────────
print(f'PyTorch: {pt_fps:6.1f} FPS | {pt_time:.1f} ms/frame')
print(f'ONNX Runtime: {onnx_fps:6.1f} FPS | {onnx_time:.1f} ms/frame')
print(f'Speedup: {onnx_fps/pt_fps:.2f}x')

PyTorch:   11.0 FPS | 91.0 ms/frame
ONNX Runtime:   18.3 FPS | 54.5 ms/frame
Speedup: 1.67x
